In [ ]:
from flask import (
    Flask, render_template, request, redirect,
    url_for, session, flash
)
import sqlite3
import os
from werkzeug.security import generate_password_hash, check_password_hash
from werkzeug.utils import secure_filename

app = Flask(__name__)
app.secret_key = "mydiary-secret-key-change-this"

DATABASE = "diary.db"
UPLOAD_FOLDER = os.path.join("static", "uploads")
ALLOWED_EXTENSIONS = {"png", "jpg", "jpeg", "gif", "webp"}

app.config["UPLOAD_FOLDER"] = UPLOAD_FOLDER
app.config["MAX_CONTENT_LENGTH"] = 5 * 1024 * 1024

os.makedirs(UPLOAD_FOLDER, exist_ok=True)


# --------------------------------------------------
# DATABASE
# --------------------------------------------------

def get_db():
    conn = sqlite3.connect(DATABASE)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    conn = get_db()

    conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            email TEXT UNIQUE NOT NULL,
            password TEXT NOT NULL
        )
    """)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS entries (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER NOT NULL,
            title TEXT NOT NULL,
            content TEXT NOT NULL,
            image TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY(user_id) REFERENCES users(id)
        )
    """)

    conn.commit()
    conn.close()


# --------------------------------------------------
# HELPERS
# --------------------------------------------------

def allowed_file(filename):
    return (
        "." in filename and
        filename.rsplit(".", 1)[1].lower() in ALLOWED_EXTENSIONS
    )


def logged_in():
    return "user_id" in session


# --------------------------------------------------
# HOME
# --------------------------------------------------

@app.route("/")
def index():
    if logged_in():
        return redirect(url_for("dashboard"))

    return render_template("login.html")


# --------------------------------------------------
# REGISTER
# --------------------------------------------------

@app.route("/register", methods=["GET", "POST"])
def register():

    if request.method == "POST":

        username = request.form.get("username", "").strip()
        email = request.form.get("email", "").strip()
        password = request.form.get("password", "")
        confirm_password = request.form.get("confirm_password", "")

        if not username or not email or not password:
            flash("All fields are required.", "error")
            return redirect(url_for("register"))

        if len(password) < 6:
            flash("Password must contain at least 6 characters.", "error")
            return redirect(url_for("register"))

        if password != confirm_password:
            flash("Passwords do not match.", "error")
            return redirect(url_for("register"))

        hashed_password = generate_password_hash(password)

        conn = get_db()

        try:
            conn.execute(
                """
                INSERT INTO users(username, email, password)
                VALUES (?, ?, ?)
                """,
                (username, email, hashed_password)
            )

            conn.commit()

        except sqlite3.IntegrityError:
            conn.close()
            flash("Username or email already exists.", "error")
            return redirect(url_for("register"))

        conn.close()

        flash("Registration successful. Please login.", "success")
        return redirect(url_for("index"))

    return render_template("register.html")


# --------------------------------------------------
# LOGIN
# --------------------------------------------------

@app.route("/login", methods=["GET", "POST"])
def login():

    if request.method == "POST":

        email = request.form.get("email", "").strip()
        password = request.form.get("password", "")

        conn = get_db()

        user = conn.execute(
            "SELECT * FROM users WHERE email = ?",
            (email,)
        ).fetchone()

        conn.close()

        if user and check_password_hash(user["password"], password):

            session.clear()

            session["user_id"] = user["id"]
            session["username"] = user["username"]

            return redirect(url_for("dashboard"))

        flash("Invalid email or password.", "error")

    return render_template("login.html")


# --------------------------------------------------
# LOGOUT
# --------------------------------------------------

@app.route("/logout")
def logout():

    session.clear()

    flash("You have been logged out.", "success")

    return redirect(url_for("index"))


# --------------------------------------------------
# DASHBOARD
# --------------------------------------------------

@app.route("/dashboard")
def dashboard():

    if not logged_in():
        return redirect(url_for("index"))

    conn = get_db()

    entries = conn.execute(
        """
        SELECT *
        FROM entries
        WHERE user_id = ?
        ORDER BY created_at DESC
        """,
        (session["user_id"],)
    ).fetchall()

    conn.close()

    return render_template(
        "dashboard.html",
        entries=entries
    )


# --------------------------------------------------
# CREATE ENTRY
# --------------------------------------------------

@app.route("/create", methods=["GET", "POST"])
def create():

    if not logged_in():
        return redirect(url_for("index"))

    if request.method == "POST":

        title = request.form.get("title", "").strip()
        content = request.form.get("content", "").strip()

        if not title or not content:
            flash("Title and content are required.", "error")
            return redirect(url_for("create"))

        image_name = None

        image = request.files.get("image")

        if image and image.filename:

            if not allowed_file(image.filename):
                flash("Invalid image format.", "error")
                return redirect(url_for("create"))

            image_name = secure_filename(image.filename)

            # Prevent duplicate filenames
            base, extension = os.path.splitext(image_name)

            counter = 1
            original_name = image_name

            while os.path.exists(
                os.path.join(app.config["UPLOAD_FOLDER"], image_name)
            ):
                image_name = f"{base}_{counter}{extension}"
                counter += 1

            image.save(
                os.path.join(
                    app.config["UPLOAD_FOLDER"],
                    image_name
                )
            )

        conn = get_db()

        conn.execute(
            """
            INSERT INTO entries(user_id, title, content, image)
            VALUES (?, ?, ?, ?)
            """,
            (
                session["user_id"],
                title,
                content,
                image_name
            )
        )

        conn.commit()
        conn.close()

        flash("Diary entry created successfully.", "success")

        return redirect(url_for("dashboard"))

    return render_template("create.html")


# --------------------------------------------------
# VIEW ENTRY
# --------------------------------------------------

@app.route("/entry/<int:entry_id>")
def view_entry(entry_id):

    if not logged_in():
        return redirect(url_for("index"))

    conn = get_db()

    entry = conn.execute(
        """
        SELECT *
        FROM entries
        WHERE id = ? AND user_id = ?
        """,
        (entry_id, session["user_id"])
    ).fetchone()

    conn.close()

    if not entry:
        flash("Entry not found.", "error")
        return redirect(url_for("dashboard"))

    return render_template(
        "view.html",
        entry=entry
    )


# --------------------------------------------------
# EDIT ENTRY
# --------------------------------------------------

@app.route("/edit/<int:entry_id>", methods=["GET", "POST"])
def edit(entry_id):

    if not logged_in():
        return redirect(url_for("index"))

    conn = get_db()

    entry = conn.execute(
        """
        SELECT *
        FROM entries
        WHERE id = ? AND user_id = ?
        """,
        (entry_id, session["user_id"])
    ).fetchone()

    if not entry:
        conn.close()
        flash("Entry not found.", "error")
        return redirect(url_for("dashboard"))

    if request.method == "POST":

        title = request.form.get("title", "").strip()
        content = request.form.get("content", "").strip()

        if not title or not content:
            conn.close()
            flash("Title and content are required.", "error")
            return redirect(url_for("edit", entry_id=entry_id))

        image_name = entry["image"]

        image = request.files.get("image")

        if image and image.filename:

            if not allowed_file(image.filename):
                conn.close()
                flash("Invalid image format.", "error")
                return redirect(url_for("edit", entry_id=entry_id))

            image_name = secure_filename(image.filename)

            base, extension = os.path.splitext(image_name)

            counter = 1

            while os.path.exists(
                os.path.join(app.config["UPLOAD_FOLDER"], image_name)
            ):
                image_name = f"{base}_{counter}{extension}"
                counter += 1

            image.save(
                os.path.join(
                    app.config["UPLOAD_FOLDER"],
                    image_name
                )
            )

        conn.execute(
            """
            UPDATE entries
            SET title = ?,
                content = ?,
                image = ?,
                updated_at = CURRENT_TIMESTAMP
            WHERE id = ? AND user_id = ?
            """,
            (
                title,
                content,
                image_name,
                entry_id,
                session["user_id"]
            )
        )

        conn.commit()
        conn.close()

        flash("Entry updated successfully.", "success")

        return redirect(
            url_for("view_entry", entry_id=entry_id)
        )

    conn.close()

    return render_template(
        "edit.html",
        entry=entry
    )


# --------------------------------------------------
# DELETE ENTRY
# --------------------------------------------------

@app.route("/delete/<int:entry_id>", methods=["POST"])
def delete(entry_id):

    if not logged_in():
        return redirect(url_for("index"))

    conn = get_db()

    entry = conn.execute(
        """
        SELECT image
        FROM entries
        WHERE id = ? AND user_id = ?
        """,
        (entry_id, session["user_id"])
    ).fetchone()

    if entry:

        if entry["image"]:

            image_path = os.path.join(
                app.config["UPLOAD_FOLDER"],
                entry["image"]
            )

            if os.path.exists(image_path):
                os.remove(image_path)

        conn.execute(
            """
            DELETE FROM entries
            WHERE id = ? AND user_id = ?
            """,
            (entry_id, session["user_id"])
        )

        conn.commit()

        flash("Entry deleted successfully.", "success")

    else:
        flash("Entry not found.", "error")

    conn.close()

    return redirect(url_for("dashboard"))


# --------------------------------------------------
# RUN APPLICATION
# --------------------------------------------------

if __name__ == "__main__":
    init_db()
    app.run(debug=True)

In [ ]:
<!DOCTYPE html>
<html lang="en">

<head>
    <meta charset="UTF-8">

    <meta name="viewport"
          content="width=device-width, initial-scale=1.0">

    <title>MyDiary</title>

    <link rel="stylesheet"
          href="{{ url_for('static', filename='style.css') }}">
</head>

<body>

<nav class="navbar">

    <a href="{{ url_for('dashboard') }}"
       class="logo">
        MyDiary
    </a>

    {% if session.get("user_id") %}

    <div class="nav-links">

        <a href="{{ url_for('dashboard') }}">
            Dashboard
        </a>

        <a href="{{ url_for('create') }}">
            New Entry
        </a>

        <a href="{{ url_for('logout') }}">
            Logout
        </a>

    </div>

    {% endif %}

</nav>


<div class="container">

    {% with messages = get_flashed_messages(with_categories=true) %}

        {% if messages %}

            {% for category, message in messages %}

                <div class="alert {{ category }}">
                    {{ message }}
                </div>

            {% endfor %}

        {% endif %}

    {% endwith %}


    {% block content %}

    {% endblock %}

</div>


<script src="{{ url_for('static', filename='script.js') }}"></script>

</body>
</html>

In [ ]:
{% extends "base.html" %}

{% block content %}

<div class="auth-container">

    <div class="auth-card">

        <h1>MyDiary</h1>

        <p class="subtitle">
            Your personal space for memories.
        </p>

        <form method="POST"
              action="{{ url_for('login') }}">

            <label>Email</label>

            <input
                type="email"
                name="email"
                placeholder="Enter your email"
                required
            >

            <label>Password</label>

            <input
                type="password"
                name="password"
                placeholder="Enter your password"
                required
            >

            <button type="submit">
                Login
            </button>

        </form>

        <p>
            Don't have an account?

            <a href="{{ url_for('register') }}">
                Register
            </a>
        </p>

    </div>

</div>

{% endblock %}

In [ ]:
{% extends "base.html" %}

{% block content %}

<div class="auth-container">

    <div class="auth-card">

        <h1>Create Account</h1>

        <form method="POST"
              action="{{ url_for('register') }}">

            <label>Username</label>

            <input
                type="text"
                name="username"
                minlength="3"
                maxlength="30"
                required
            >

            <label>Email</label>

            <input
                type="email"
                name="email"
                required
            >

            <label>Password</label>

            <input
                type="password"
                name="password"
                minlength="6"
                required
            >

            <label>Confirm Password</label>

            <input
                type="password"
                name="confirm_password"
                minlength="6"
                required
            >

            <button type="submit">
                Create Account
            </button>

        </form>

        <p>
            Already have an account?

            <a href="{{ url_for('index') }}">
                Login
            </a>
        </p>

    </div>

</div>

{% endblock %}

In [ ]:
{% extends "base.html" %}

{% block content %}

<div class="dashboard-header">

    <div>

        <h1>
            Welcome, {{ session["username"] }} 👋
        </h1>

        <p>
            Your personal diary
        </p>

    </div>

    <a class="button"
       href="{{ url_for('create') }}">
        + New Entry
    </a>

</div>


{% if entries %}

<div class="entries-grid">

    {% for entry in entries %}

    <div class="entry-card">

        {% if entry["image"] %}

        <img
            src="{{ url_for('static',
                            filename='uploads/' + entry['image']) }}"
            alt="Diary image"
        >

        {% endif %}

        <div class="entry-content">

            <h2>
                {{ entry["title"] }}
            </h2>

            <p class="date">
                {{ entry["created_at"] }}
            </p>

            <p>
                {{ entry["content"][:180] }}
                {% if entry["content"]|length > 180 %}
                    ...
                {% endif %}
            </p>

            <div class="actions">

                <a href="{{ url_for('view_entry',
                                    entry_id=entry['id']) }}">
                    View
                </a>

                <a href="{{ url_for('edit',
                                    entry_id=entry['id']) }}">
                    Edit
                </a>

                <form
                    method="POST"
                    action="{{ url_for('delete',
                                       entry_id=entry['id']) }}"
                    onsubmit="return confirmDelete();"
                >

                    <button type="submit"
                            class="delete-button">
                        Delete
                    </button>

                </form>

            </div>

        </div>

    </div>

    {% endfor %}

</div>

{% else %}

<div class="empty-state">

    <h2>No diary entries yet.</h2>

    <p>
        Start writing your first memory.
    </p>

    <a class="button"
       href="{{ url_for('create') }}">
        Create Entry
    </a>

</div>

{% endif %}

{% endblock %}

In [ ]:
{% extends "base.html" %}

{% block content %}

<div class="form-container">

    <h1>New Diary Entry</h1>

    <form
        method="POST"
        enctype="multipart/form-data"
        action="{{ url_for('create') }}"
    >

        <label>Title</label>

        <input
            type="text"
            name="title"
            maxlength="100"
            placeholder="Give your entry a title"
            required
        >

        <label>Write your thoughts</label>

        <textarea
            name="content"
            rows="12"
            maxlength="10000"
            placeholder="Dear diary..."
            required
        ></textarea>

        <label>Upload Image</label>

        <input
            type="file"
            name="image"
            accept=".jpg,.jpeg,.png,.gif,.webp"
        >

        <div class="form-buttons">

            <button type="submit">
                Save Entry
            </button>

            <a href="{{ url_for('dashboard') }}">
                Cancel
            </a>

        </div>

    </form>

</div>

{% endblock %}

In [ ]:
{% extends "base.html" %}

{% block content %}

<div class="form-container">

    <h1>Edit Entry</h1>

    <form
        method="POST"
        enctype="multipart/form-data"
        action="{{ url_for('edit',
                           entry_id=entry['id']) }}"
    >

        <label>Title</label>

        <input
            type="text"
            name="title"
            maxlength="100"
            value="{{ entry['title'] }}"
            required
        >

        <label>Content</label>

        <textarea
            name="content"
            rows="12"
            maxlength="10000"
            required
        >{{ entry['content'] }}</textarea>

        {% if entry["image"] %}

        <div class="current-image">

            <p>Current image:</p>

            <img
                src="{{ url_for('static',
                                filename='uploads/' + entry['image']) }}"
                alt="Current diary image"
            >

        </div>

        {% endif %}

        <label>Replace Image</label>

        <input
            type="file"
            name="image"
            accept=".jpg,.jpeg,.png,.gif,.webp"
        >

        <button type="submit">
            Update Entry
        </button>

    </form>

</div>

{% endblock %}

In [ ]:
{% extends "base.html" %}

{% block content %}

<div class="single-entry">

    <h1>{{ entry["title"] }}</h1>

    <p class="date">
        Created: {{ entry["created_at"] }}
    </p>

    {% if entry["image"] %}

    <img
        class="entry-image"
        src="{{ url_for('static',
                        filename='uploads/' + entry['image']) }}"
        alt="Diary image"
    >

    {% endif %}

    <div class="entry-text">

        {{ entry["content"] }}

    </div>

    <div class="actions">

        <a class="button"
           href="{{ url_for('edit',
                            entry_id=entry['id']) }}">
            Edit
        </a>

        <a href="{{ url_for('dashboard') }}">
            Back to Dashboard
        </a>

    </div>

</div>

{% endblock %}

In [ ]:
* {
    box-sizing: border-box;
    margin: 0;
    padding: 0;
}

body {
    font-family: Arial, sans-serif;
    background: #f5f5f5;
    color: #333;
}

.navbar {
    background: #222;
    color: white;
    padding: 18px 8%;
    display: flex;
    justify-content: space-between;
    align-items: center;
}

.logo {
    color: white;
    text-decoration: none;
    font-size: 24px;
    font-weight: bold;
}

.nav-links {
    display: flex;
    gap: 25px;
}

.nav-links a {
    color: white;
    text-decoration: none;
}

.container {
    width: 85%;
    max-width: 1200px;
    margin: 40px auto;
}

.auth-container {
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 75vh;
}

.auth-card {
    width: 400px;
    background: white;
    padding: 40px;
    border-radius: 12px;
    box-shadow: 0 5px 25px rgba(0,0,0,0.1);
}

.auth-card h1 {
    text-align: center;
    margin-bottom: 10px;
}

.subtitle {
    text-align: center;
    margin-bottom: 30px;
    color: #777;
}

form {
    display: flex;
    flex-direction: column;
    gap: 10px;
}

label {
    font-weight: bold;
    margin-top: 10px;
}

input,
textarea {
    width: 100%;
    padding: 12px;
    border: 1px solid #ccc;
    border-radius: 6px;
    font-size: 15px;
}

textarea {
    resize: vertical;
}

button,
.button {
    display: inline-block;
    background: #222;
    color: white;
    padding: 12px 20px;
    border: none;
    border-radius: 6px;
    text-decoration: none;
    cursor: pointer;
    margin-top: 10px;
}

button:hover,
.button:hover {
    opacity: 0.85;
}

.auth-card button {
    width: 100%;
}

.auth-card p {
    text-align: center;
    margin-top: 20px;
}

.auth-card a {
    color: #222;
    font-weight: bold;
}

.dashboard-header {
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 30px;
}

.dashboard-header p {
    color: #777;
    margin-top: 5px;
}

.entries-grid {
    display: grid;
    grid-template-columns: repeat(
        auto-fit,
        minmax(280px, 1fr)
    );
    gap: 25px;
}

.entry-card {
    background: white;
    border-radius: 10px;
    overflow: hidden;
    box-shadow: 0 4px 15px rgba(0,0,0,0.08);
}

.entry-card img {
    width: 100%;
    height: 180px;
    object-fit: cover;
}

.entry-content {
    padding: 20px;
}

.entry-content h2 {
    margin-bottom: 8px;
}

.date {
    color: #888;
    font-size: 13px;
    margin-bottom: 15px;
}

.actions {
    display: flex;
    align-items: center;
    gap: 15px;
    margin-top: 20px;
}

.actions a {
    color: #222;
    text-decoration: none;
    font-weight: bold;
}

.actions form {
    display: inline;
}

.delete-button {
    background: #c0392b;
    margin: 0;
}

.form-container {
    background: white;
    max-width: 800px;
    margin: auto;
    padding: 35px;
    border-radius: 12px;
    box-shadow: 0 5px 20px rgba(0,0,0,0.08);
}

.form-container h1 {
    margin-bottom: 25px;
}

.form-buttons {
    display: flex;
    gap: 15px;
    align-items: center;
}

.form-buttons a {
    color: #222;
    text-decoration: none;
}

.single-entry {
    background: white;
    max-width: 850px;
    margin: auto;
    padding: 40px;
    border-radius: 12px;
}

.entry-image {
    width: 100%;
    max-height: 500px;
    object-fit: cover;
    border-radius: 10px;
    margin: 20px 0;
}

.entry-text {
    white-space: pre-wrap;
    line-height: 1.8;
    font-size: 17px;
    margin-top: 25px;
}

.current-image img {
    width: 200px;
    border-radius: 8px;
}

.empty-state {
    text-align: center;
    background: white;
    padding: 80px 20px;
    border-radius: 12px;
}

.empty-state p {
    margin: 15px 0 25px;
    color: #777;
}

.alert {
    padding: 15px;
    margin-bottom: 20px;
    border-radius: 6px;
}

.alert.success {
    background: #d4edda;
    color: #155724;
}

.alert.error {
    background: #f8d7da;
    color: #721c24;
}

@media (max-width: 700px) {

    .navbar {
        padding: 15px 5%;
    }

    .nav-links {
        gap: 10px;
        font-size: 14px;
    }

    .container {
        width: 92%;
    }

    .dashboard-header {
        flex-direction: column;
        align-items: flex-start;
        gap: 20px;
    }

    .auth-card,
    .form-container,
    .single-entry {
        padding: 25px;
    }
}

In [ ]:
function confirmDelete() {
    return confirm(
        "Are you sure you want to delete this diary entry?"
    );
}


// Automatically hide flash messages

setTimeout(function () {

    const alerts = document.querySelectorAll(".alert");

    alerts.forEach(function(alert) {
        alert.style.opacity = "0";
        alert.style.transition = "opacity 0.5s";

        setTimeout(function() {
            alert.remove();
        }, 500);
    });

}, 4000);